# SynBioCrow 2.3 — Route Preservation Audit
Offline audit only. No retrosynthesis backends are rerun. The notebook checks whether routes found by individual engines remain present in the union graph and distinguishes graph-loss defects from route-enumeration truncation/order effects.


In [ ]:
import subprocess,sys,shutil,json,hashlib,zipfile
from pathlib import Path
from google.colab import drive, files

drive.mount("/content/drive",force_remount=False)
REPO=Path("/content/SynBioCrow_route_audit")
if REPO.exists():
    shutil.rmtree(REPO)

subprocess.run(["git","clone","--depth","1","--branch","develop/2.3",
                "https://github.com/theiman112860/SynBioCrow.git",str(REPO)],check=True)
subprocess.run([sys.executable,"-m","pip","install","-q","-e",str(REPO)],check=True)

PANEL=REPO/"benchmarks/2.3/manuscript_panel_v2.json"
STATE_OLD=Path("/content/drive/MyDrive/SynBioCrow/benchmark/2.3/manuscript_panel_v2/state")
STATE_RECOVERY=Path("/content/drive/MyDrive/SynBioCrow/benchmark/2.3/manuscript_panel_v2_general_completion/state")
OUT=Path("/content/drive/MyDrive/SynBioCrow/benchmark/2.3/route_preservation_audit")
OUT.mkdir(parents=True,exist_ok=True)

for p in (STATE_OLD,STATE_RECOVERY):
    print("STATE",p,"exists=",p.exists())

report=OUT/"route_preservation_audit.json"
cmd=[
    sys.executable,str(REPO/"scripts/audit_route_preservation_2_3.py"),
    "--panel",str(PANEL),
    "--state-root",str(STATE_RECOVERY),
    "--state-root",str(STATE_OLD),
    "--output",str(report),
    "--max-route-steps","8",
    "--max-routes","100",
]
print("$"," ".join(cmd),flush=True)
p=subprocess.run(cmd,cwd=REPO,text=True,stdout=subprocess.PIPE,stderr=subprocess.STDOUT)
print(p.stdout,flush=True)

# A graph-monotonicity defect deliberately returns rc=1 so it cannot be overlooked.
if p.returncode not in (0,1):
    raise subprocess.CalledProcessError(p.returncode,cmd,output=p.stdout)

payload=json.loads(report.read_text())
print(json.dumps({
    "targets_with_individual_routes":payload["targets_with_individual_routes"],
    "total_graph_monotonicity_violations":payload["total_graph_monotonicity_violations"],
    "total_enumeration_losses":payload["total_enumeration_losses"],
    "truth_accessed":payload["truth_accessed"],
},indent=2))

bundle=OUT/"SynBioCrow_2_3_ROUTE_PRESERVATION_AUDIT.zip"
with zipfile.ZipFile(bundle,"w",zipfile.ZIP_DEFLATED) as z:
    z.write(report,report.name)
    z.write(REPO/"scripts/audit_route_preservation_2_3.py","audit_route_preservation_2_3.py")
    z.write(REPO/"tests/test_route_preservation_2_3.py","test_route_preservation_2_3.py")
h=hashlib.sha256(bundle.read_bytes()).hexdigest()
print("BUNDLE",bundle)
print("SHA256",h)

if payload["total_graph_monotonicity_violations"]:
    print("RESULT: GRAPH MONOTONICITY DEFECT DETECTED")
elif payload["total_enumeration_losses"]:
    print("RESULT: NO EDGE LOSS; ROUTE ENUMERATION/TRUNCATION EFFECT DETECTED")
else:
    print("RESULT: INDIVIDUAL ROUTES PRESERVED IN ENSEMBLE OUTPUT")

files.download(str(bundle))
